# Scaling study — Qwen2.5 7B / 14B on a free Colab T4

**What this is for.** The CPU-only 3B results are the project's primary contribution. This notebook adds a **scaling curve** on top of them: 3B → 7B → 14B, with everything except model scale held constant.

The point is not a bigger accuracy number. It is the **shape of the curve** relative to the clinical safety bars (low 70% / medium 80% / high 90%). A single model's ceiling is an observation; a curve is an *extrapolation* — it says whether the bars are reachable by scaling **at all**.

**Runtime — Colab free tier caps sessions at ~12 h and disconnects when idle:**

| Model | Q4_K_M size | Fits T4 (16 GB)? | All policies |
|---|---|---|---|
| Qwen2.5-7B | ~4.5 GB | yes | 11–18 h (~2 sessions) |
| Qwen2.5-14B | ~9 GB | yes, tight | 22–35 h (~3+ sessions) |
| Qwen2.5-32B | ~19 GB | **no** — exceeds VRAM | — |

**Everything here is resumable.** Logs live on Google Drive and `run_experiment.py` skips question-ids already present, so a disconnect costs minutes, not the run. Re-run the notebook top-to-bottom after any disconnect.

**Do not skip the smoke test (step 6).** Qwen uses ChatML (`<|im_start|>`), not Llama's `[INST]`. The wrong markup does not raise an error — it silently degrades the answers, and you would be measuring the prompt rather than the model. The smoke test is what stands between you and a wasted 22 hours.

## 1. Check the GPU

Runtime → Change runtime type → **T4 GPU**. If this cell shows no GPU, fix that before continuing — everything below assumes CUDA.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total,memory.free --format=csv

## 2. Mount Drive

Drive is what makes this resumable: the model weights, the indexes and the logs all
persist there across sessions, so a disconnect never destroys work.

**One-time setup on your own machine.** Supply retrieval one of two ways:

**A — prebuilt indexes (preferred, 1.81 GB).** No on-box rebuild, and it skips the
assemble stage, which is the only step here that can run out of RAM. Upload:

```
MyDrive/medrag/indexes/bm25_medcorp_tp.pkl              (786 MB)
MyDrive/medrag/indexes/faiss_medcorp_tp/faiss.index     (624 MB)
MyDrive/medrag/indexes/faiss_medcorp_tp/chunks.pkl      (401 MB)
```

Do **not** upload `indexes/faiss_medcorp_tp/_shards/` (213 files, 639 MB) — those are
embed checkpoints used only when rebuilding. The corpus JSONL is not needed either:
`chunks.pkl` already carries the chunk text.

**B — corpus only (fallback, 412 MB).** Cheaper upload, but costs ~25 min of GPU
rebuild and carries the OOM risk:

```
MyDrive/medrag/data/corpora/medcorp_tp.jsonl
```

The notebook detects which you provided and takes the right path.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
DRIVE = '/content/drive/MyDrive/medrag'
for sub in ('logs', 'models', 'indexes', 'data/corpora'):
    os.makedirs(f'{DRIVE}/{sub}', exist_ok=True)

# Two ways to supply retrieval, checked in order of preference:
#
#   A. PREBUILT INDEXES on Drive (1.81 GB) -- preferred. No rebuild, and it skips
#      the assemble stage, which is the only step here that can OOM.
#   B. CORPUS JSONL on Drive (412 MB) -- fallback. Rebuilt on the GPU (~25 min).
#
# With prebuilt indexes the corpus is not needed at runtime: chunks.pkl already
# carries the chunk text.
corpus = f'{DRIVE}/data/corpora/medcorp_tp.jsonl'
HAVE_CORPUS = os.path.exists(corpus)
HAVE_INDEXES = (os.path.exists(f'{DRIVE}/indexes/bm25_medcorp_tp.pkl') and
                os.path.exists(f'{DRIVE}/indexes/faiss_medcorp_tp/faiss.index') and
                os.path.exists(f'{DRIVE}/indexes/faiss_medcorp_tp/chunks.pkl'))

print('prebuilt indexes on Drive:', HAVE_INDEXES)
print('corpus jsonl on Drive    :', HAVE_CORPUS,
      f'({round(os.path.getsize(corpus) / 1e6)} MB)' if HAVE_CORPUS else '')

assert HAVE_INDEXES or HAVE_CORPUS, (
    'Neither prebuilt indexes nor the corpus were found on Drive.\n'
    'Upload EITHER (preferred, 1.81 GB):\n'
    '  MyDrive/medrag/indexes/bm25_medcorp_tp.pkl\n'
    '  MyDrive/medrag/indexes/faiss_medcorp_tp/faiss.index\n'
    '  MyDrive/medrag/indexes/faiss_medcorp_tp/chunks.pkl\n'
    'OR (fallback, 412 MB):\n'
    '  MyDrive/medrag/data/corpora/medcorp_tp.jsonl'
)


## 3. Clone the repo and install

`llama-cpp-python` is built **with CUDA** here. That single flag is the whole GPU story for this project: llama-cpp under CUDA still exposes `_scores` (entropy gate) and `logprobs=2` (margin gate), so the gates work unchanged.

This is why the backend is not swapped for vLLM or TGI — those serve faster but do not expose per-token logits usefully, which would **break the gates** and with them the entire contribution being measured.

The build takes ~5 minutes.

In [ ]:
%cd /content
![ -d FinalProject_KCL ] || git clone https://github.com/Ideapersie/FinalProject_KCL.git
%cd /content/FinalProject_KCL

import os, sys, glob, shutil, subprocess, importlib, torch

# llama-cpp-python is PINNED to 0.3.28 — the version the gates are proven against. The
# entropy gate reads `_scores` and the margin gate needs `logprobs=2`; a release that
# moves either returns None (P5 silently retrieves everything). Do not float it.
#
# Colab is on CUDA 12.8, for which no prebuilt wheel exists, so a from-source CUDA
# build (~1 h) would otherwise repeat every session. We build/download the wheel ONCE,
# cache the .whl to Drive, and reinstall from that cache (~10 s) on every later session.

WHEELS_LOCAL = '/content/wheels'
os.makedirs(WHEELS_LOCAL, exist_ok=True)
CACHE = f'{DRIVE}/wheels' if 'DRIVE' in globals() else None   # DRIVE set by the mount cell
if CACHE:
    os.makedirs(CACHE, exist_ok=True)

# llama-cpp-python's OWN runtime deps, installed explicitly because every wheel
# install below passes --no-deps to keep pip away from numpy/torch and the CUDA
# stack. Colab ships jinja2, numpy and typing-extensions but NOT diskcache, and
# llama_cpp/llama_cache.py imports it at module import time. Without this line
# `import llama_cpp` raises ModuleNotFoundError, which the old check reported as
# "does not offload" — so a perfectly good 1.5 GB cached CUDA wheel was discarded,
# and the from-source fallback then failed its own assert after ~1 hour.
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                'diskcache', 'jinja2', 'typing-extensions'], check=False)

def install_whl(path):
    """Install a wheel, then verify it imports AND offloads.

    Reports which of the three ways it failed, because they need different fixes:
    a pip failure means a bad/partial file, an import failure means a missing
    dependency, and a False from llama_supports_gpu_offload means the wheel
    really was built without CUDA.
    """
    r = subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                        '--force-reinstall', '--no-deps', path],
                       capture_output=True, text=True)
    if r.returncode != 0:
        print(f'  pip install FAILED (rc={r.returncode}) — bad or partial file:')
        print('   ', (r.stderr or r.stdout or '').strip()[-300:])
        return False

    # Fresh interpreter, so the kernel never caches a CPU-only build.
    probe = subprocess.run(
        [sys.executable, '-c',
         'from llama_cpp import llama_supports_gpu_offload as f;'
         ' import sys; sys.exit(0 if f() else 2)'],
        capture_output=True, text=True)
    if probe.returncode == 0:
        return True
    if probe.returncode == 2:
        print('  imports, but reports no GPU offload — this wheel is a CPU-only build')
    else:
        last = (probe.stderr or '').strip().splitlines()
        print(f'  installed but IMPORT FAILED: {last[-1] if last else "unknown"}')
        print('    (a missing dependency, not a bad wheel — do not rebuild)')
    return False

CUDA = torch.version.cuda
exact = 'cu' + CUDA.replace('.', '') if CUDA else None
tags, seen = [], set()
for t in [exact, 'cu125', 'cu124', 'cu123', 'cu122', 'cu121']:
    if t and t not in seen:
        seen.add(t); tags.append(t)
print('torch CUDA:', CUDA)

installed = False

# 1) cached working wheel from a previous session
if CACHE:
    for whl in sorted(glob.glob(f'{CACHE}/llama_cpp_python-*.whl')):
        print('cached Drive wheel:', os.path.basename(whl))
        if install_whl(whl):
            print('  offloads to GPU ✓ (installed from Drive cache)'); installed = True; break
        print('  cached wheel does not offload here — re-acquiring')

# 2) prebuilt abetlen wheel, newest tag first; cache the first that works
if not installed:
    for tag in tags:
        for w in glob.glob(f'{WHEELS_LOCAL}/*.whl'):
            os.remove(w)
        r = subprocess.run([sys.executable, '-m', 'pip', 'download', '-q', '--no-deps',
            'llama-cpp-python==0.3.28', '-d', WHEELS_LOCAL,
            '--index-url', f'https://abetlen.github.io/llama-cpp-python/whl/{tag}'])
        got = sorted(glob.glob(f'{WHEELS_LOCAL}/llama_cpp_python-*.whl'))
        if r.returncode == 0 and got and install_whl(got[-1]):
            if CACHE:
                shutil.copy(got[-1], CACHE); print(f'  cached to Drive: {os.path.basename(got[-1])}')
            print(f'  {tag}: offloads to GPU ✓'); installed = True; break
        print(f'  {tag}: no usable prebuilt wheel')

# 3) from-source build via `pip wheel` (produces a reusable .whl), cached
if not installed:
    print('no prebuilt wheel for this CUDA — building from source ONCE (~1 h); result cached...')
    for w in glob.glob(f'{WHEELS_LOCAL}/*.whl'):
        os.remove(w)
    subprocess.run('CMAKE_ARGS="-DGGML_CUDA=on" pip wheel -q llama-cpp-python==0.3.28 '
                   f'--no-deps -w {WHEELS_LOCAL}', shell=True)
    got = sorted(glob.glob(f'{WHEELS_LOCAL}/llama_cpp_python-*.whl'))
    assert got, 'source build produced no wheel'
    assert install_whl(got[-1]), 'built wheel does not offload — is the T4 runtime selected?'
    if CACHE:
        shutil.copy(got[-1], CACHE); print(f'  cached to Drive: {os.path.basename(got[-1])}')
    installed = True

assert installed, 'could not obtain a GPU-capable llama-cpp-python'

!pip install -q sentence-transformers faiss-cpu rank-bm25 pydantic pyyaml datasets huggingface_hub

# Installing sentence-transformers pulls a transformers/torch whose libtorch_cuda.so
# references an NCCL symbol (ncclCommShrink) that the preinstalled nvidia-nccl-cu12
# lacks, breaking `import torch` — and thus the vector retriever — at the FIRST
# retrieval (not at startup, so the smoke test misses it). torch pins an EXACT nccl
# (torch 2.11.0+cu128 -> 2.28.9); install that. If Colab's torch changes, the pip
# conflict message names the version to pin here.
!pip install -q nvidia-nccl-cu12==2.28.9
!python -c "import torch; from sentence_transformers import SentenceTransformer; print('torch import OK', torch.__version__)"

# Make medrag_adaptive importable everywhere WITHOUT an editable install (whose PEP660
# import hook activates only at interpreter startup, so it is invisible to THIS running
# kernel). PYTHONPATH is inherited by every subprocess (run_experiment.py) and !python
# call; sys.path covers the main kernel now.
SRC = os.path.abspath('src')
os.environ['PYTHONPATH'] = SRC + os.pathsep + os.environ.get('PYTHONPATH', '')
sys.path.insert(0, SRC)
importlib.invalidate_caches()

import llama_cpp, medrag_adaptive
print('llama-cpp-python', llama_cpp.__version__)
print('medrag_adaptive importable:', medrag_adaptive.__file__)


In [ ]:
# Confirm llama-cpp actually sees the GPU. If this says False, the CUDA build
# silently fell back to CPU and a 14B run would take days rather than hours.
from llama_cpp import llama_supports_gpu_offload
print('GPU offload available:', llama_supports_gpu_offload())
assert llama_supports_gpu_offload(), 'CUDA build failed — re-run the install cell.'

## 4. Download the model weights (Q4_K_M)

**Q4_K_M for both models, matching the 3B baseline's quantisation.** This matters: a higher-fidelity quant here would confound *bigger model* with *less quantisation*, and the curve would no longer isolate scale.

Weights are cached on Drive, so this only downloads once across all sessions.

In [ ]:
# Pick ONE per session. Start with 7B — it is faster and validates the pipeline
# before you commit a 22-hour 14B run to it.
MODEL = '7b'          # '7b' or '14b'

SPECS = {
    '7b':  dict(repo='Qwen/Qwen2.5-7B-Instruct-GGUF',
                file='qwen2.5-7b-instruct-q4_k_m.gguf',
                cfg='configs/models/qwen7b.yaml'),
    '14b': dict(repo='Qwen/Qwen2.5-14B-Instruct-GGUF',
                file='qwen2.5-14b-instruct-q4_k_m.gguf',
                cfg='configs/models/qwen14b.yaml'),
}
spec = dict(SPECS[MODEL])   # COPY — mutating this must not touch SPECS.

# Qwen's GGUF repos ship Q4_K_M as multiple shards
# (`...-q4_k_m-00001-of-00002.gguf`), not one file. Resolve the real names against
# the repo listing before downloading, and download every shard.
from huggingface_hub import list_repo_files, hf_hub_download
files = [f for f in list_repo_files(spec['repo']) if f.lower().endswith('.gguf')]
q4 = sorted(f for f in files if 'q4_k_m' in f.lower())
print('Q4_K_M files in repo:', q4)
assert q4, f'no Q4_K_M gguf in {spec["repo"]} — available: {files}'
if len(q4) > 1:
    print(f'sharded across {len(q4)} files')

import os, re
os.makedirs('models', exist_ok=True)
for shard in q4:
    cached = f"{DRIVE}/models/{os.path.basename(shard)}"
    if not os.path.exists(cached):
        print('downloading', shard)
        hf_hub_download(repo_id=spec['repo'], filename=shard, local_dir=f'{DRIVE}/models')
    local = f"models/{os.path.basename(shard)}"
    if not os.path.exists(local):
        os.symlink(os.path.abspath(cached), local)

# llama.cpp is given the FIRST shard and finds the rest by naming convention.
GGUF = f"models/{os.path.basename(q4[0])}"
print(f"{GGUF}: {os.path.getsize(GGUF) / 1e9:.1f} GB"
      + (f"  (+{len(q4)-1} more shard(s))" if len(q4) > 1 else ""))

# The model YAML hard-codes the single-file name. Rewrite gguf_path to the actual
# downloaded path so BOTH the smoke test (which has no --model flag) and the runs
# read the correct location. Everything downstream now works from the YAML alone.
cfg_text = open(spec['cfg'], encoding='utf-8').read()
cfg_text = re.sub(r'gguf_path: .*', f'gguf_path: {GGUF}', cfg_text)
open(spec['cfg'], 'w', encoding='utf-8').write(cfg_text)
print('patched', spec['cfg'], '->', [l for l in cfg_text.splitlines() if 'gguf_path' in l][0].strip())

# No override needed any more; kept as an empty list so later cells still reference it.
GGUF_OVERRIDE = []


## 5. Get the indexes onto the box

If you uploaded the prebuilt indexes (option A), this just copies them to local disk
(~2 min) — local rather than the Drive mount, because FAISS and BM25 do heavy random
reads at query time and Drive FUSE is slow for that.

If you uploaded only the corpus (option B), this rebuilds on the GPU instead
(~25 min) and caches the result to Drive so later sessions skip it.

In [ ]:
import os, shutil, time

BM25 = 'indexes/bm25_medcorp_tp.pkl'
FAISS = 'indexes/faiss_medcorp_tp'
os.makedirs('indexes', exist_ok=True)
os.makedirs('data/corpora', exist_ok=True)

if HAVE_INDEXES:
    # Copy to local disk rather than reading over the Drive FUSE mount: FAISS and
    # BM25 do heavy random reads at query time and Drive is slow for that. ~2 min.
    if not os.path.exists(BM25):
        t = time.time()
        print('copying prebuilt indexes from Drive (~1.8 GB)...')
        shutil.copy(f'{DRIVE}/indexes/bm25_medcorp_tp.pkl', BM25)
        os.makedirs(FAISS, exist_ok=True)
        for f in ('faiss.index', 'chunks.pkl'):
            shutil.copy(f'{DRIVE}/indexes/faiss_medcorp_tp/{f}', f'{FAISS}/{f}')
        print(f'done in {time.time() - t:.0f}s')
    else:
        print('indexes already present locally')
else:
    # Fallback: rebuild from the corpus on the GPU (~25 min).
    print('no prebuilt indexes — rebuilding from the corpus')
    if not os.path.exists('data/corpora/medcorp_tp.jsonl'):
        # Symlink, so the DOWNLOAD stage is skipped: re-fetching 426K chunks from
        # HuggingFace when the corpus is already on Drive would be pure waste.
        os.symlink(corpus, 'data/corpora/medcorp_tp.jsonl')

    # Shards are checkpointed to disk, so a disconnect mid-embed loses at most one
    # shard -- just re-run this cell.
    print('embedding corpus on GPU (~10 min)...')
    !python scripts/build_medcorp.py --stage embed --name medcorp_tp
    print('assembling FAISS + BM25 indexes (~10 min)...')
    !python scripts/build_medcorp.py --stage assemble --name medcorp_tp

    # Cache to Drive so later sessions skip all of this.
    print('caching indexes to Drive...')
    shutil.copy(BM25, f'{DRIVE}/indexes/')
    os.makedirs(f'{DRIVE}/indexes/faiss_medcorp_tp', exist_ok=True)
    for f in ('faiss.index', 'chunks.pkl'):
        shutil.copy(f'{FAISS}/{f}', f'{DRIVE}/indexes/faiss_medcorp_tp/{f}')

for p in (BM25, f'{FAISS}/faiss.index', f'{FAISS}/chunks.pkl'):
    assert os.path.exists(p), f'missing after setup: {p}'
    print(f'  {p}  {os.path.getsize(p) / 1e6:.0f} MB')
print('indexes ready')


## 6. 🔴 SMOKE TEST — do not skip

**This is the gate. Nothing long runs until it passes.**

It checks the four things that can silently ruin a 22-hour run:

1. **Chat format** — the prompt uses Qwen's ChatML, not Llama's `[INST]`. Wrong markup degrades answers *without erroring*.
2. **Answers parse** — `extract_letter` finds a letter. If not, accuracy reads ~0% and looks like a bad model rather than a bad parser.
3. **Gates fire** — entropy and margin return real numbers, i.e. `logits_all` survived the CUDA build. If they are dead, P5 silently degenerates into retrieve-everything.
4. **Signals span the threshold** — if every query lands on one side of τ, the gate never fires. That is *expected* (it is the calibration-non-transfer finding) and is fixed offline in step 8 — but it must be caught **now**, not after the run.

In [ ]:
!python scripts/smoke_test_model.py \
  --model-config {spec['cfg']} \
  --dataset data/raw/mirage/benchmark.json \
  --bm25-index {BM25} --faiss-index {FAISS} \
  -n 20

## 6a. Latency profile — before committing to any long run

Measured on the 2026-07-25 session: Qwen answers a question in **0.1 s** closed-book
(MCQ) and **2.3 s** open-ended, but P5 costs **300 s** (MCQ) and **531 s** (open) —
including **211 s on the SKIP path**, where nothing is retrieved and the only extra work
is four 48-token gate drafts. On a T4 those drafts should cost single-digit seconds, so
roughly 200 s per question is unaccounted for.

At that rate the 200-question open-ended run needs ~30 h. That is why the first attempt
died at 35 of 200 — not a crash, just the session limit.

The suspicion is `logits_all=True`: it makes llama.cpp compute and retain the
full-vocabulary output projection for *every* context position, and Qwen's vocabulary is
152k at `n_ctx` 4096. Only the entropy gate needs it; the margin and probe gates read
`logprobs` instead. **This is a hypothesis, not a diagnosis** — that is what the cell
below is for.

Takes ~2 minutes. Read its output before running step 7.

In [ ]:
# Times each P5 stage in isolation: logits_all on vs off, short prompt vs long.
# The four gate calls are what P5 adds over P3, so if P5 is slow the cost is here.
import gc, sys, time
sys.path.insert(0, 'src')

from medrag_adaptive.config import load_config
from medrag_adaptive.models.llama_backend import LlamaBackend
from medrag_adaptive.models.prompts import (build_closed_book_prompt,
                                            build_draft_prompt, set_chat_format)

_cfg = load_config(base='configs/base.yaml', model=spec['cfg'],
                   policy='configs/policies/p5_gated_entropy.yaml',
                   experiment='configs/experiments/pubmedqa_open.yaml')
set_chat_format(_cfg.model.chat_format)
print(f"gguf={_cfg.model.gguf_path}  n_ctx={_cfg.model.n_ctx}  "
      f"gpu_layers={_cfg.model.n_gpu_layers}  n_batch={_cfg.hardware.n_batch}")

Q  = ("A patient develops anaphylactic shock after a bee sting. "
      "What is the first-line drug?")
CH = {"A": "Diphenhydramine", "B": "Epinephrine",
      "C": "Hydrocortisone", "D": "Salbutamol"}

SHORT = build_draft_prompt(Q, CH)
# Stand-in for a 5-chunk RAG prompt, so the long-context cost is measured too.
_filler = ("Anaphylaxis is treated with intramuscular epinephrine without delay. " * 130)
LONG = build_closed_book_prompt("CONTEXT:\n" + _filler + "\n\n" + Q, CH)
print(f"prompt tokens: short~{len(SHORT)//4}  long~{len(LONG)//4}")


def _t(label, fn, *a, **k):
    t0 = time.perf_counter()
    fn(*a, **k)
    dt = time.perf_counter() - t0
    print(f'  {label:<44}{dt:8.2f} s')
    return dt


results = {}
for logits_all in (True, False):
    print(f"\n===== logits_all={logits_all} =====")
    t0 = time.perf_counter()
    llm = LlamaBackend(
        gguf_path=_cfg.model.gguf_path, n_ctx=_cfg.model.n_ctx,
        n_threads=_cfg.hardware.n_threads, n_batch=_cfg.hardware.n_batch,
        logits_all=logits_all, temperature=0.0, max_new_tokens=256,
        seed=42, verbose=False, chat_format=_cfg.model.chat_format,
        n_gpu_layers=_cfg.model.n_gpu_layers)
    print(f'  {"model load":<44}{time.perf_counter()-t0:8.2f} s')

    d = {}
    d['entropy draft  (short, 48 tok)'] = _t('entropy draft  (short, 48 tok)',
                                             llm.draft, SHORT, 48)
    d['margin logprobs(short, 48 tok)'] = _t('margin logprobs(short, 48 tok)',
                                             llm.get_top2_logprobs, SHORT, 48)
    d['probe draft x2 (short, 48 tok)'] = _t('probe draft x2 (short, 48 tok)',
                                             lambda: (llm.draft(SHORT, 48),
                                                      llm.draft(SHORT, 48)))
    d['answer         (short)'] = _t('answer         (short)', llm.answer, SHORT, 64)
    d['entropy draft  (LONG, 48 tok)'] = _t('entropy draft  (LONG, 48 tok)',
                                            llm.draft, LONG, 48)
    d['answer         (LONG)'] = _t('answer         (LONG)', llm.answer, LONG, 256)
    results[logits_all] = d

    gate_cost = (d['entropy draft  (short, 48 tok)']
                 + d['margin logprobs(short, 48 tok)']
                 + d['probe draft x2 (short, 48 tok)'])
    print(f'  -> gate total (the P5 SKIP-path overhead): {gate_cost:.2f} s')

    llm.close()
    del llm
    gc.collect()

print("\n===== VERDICT =====")
on = sum(results[True].values())
off = sum(results[False].values())
print(f"total with logits_all=True : {on:8.2f} s")
print(f"total with logits_all=False: {off:8.2f} s")
if off > 0 and on / off > 3:
    print(f"=> logits_all costs {on/off:.1f}x. It is the dominant cost; a per-call fix")
    print("   (full logits ONLY for the entropy gate) is worth doing before the long run.")
else:
    print("=> logits_all is NOT the dominant cost. Do not build a fix around it;")
    print("   profile retrieval next (BM25Okapi.get_scores is pure Python and linear")
    print("   in corpus size, and medcorp is large).")

skip_measured = 211.0   # observed Qwen P5 MCQ skip-path latency, 2026-07-25
print(f"\nobserved P5 skip path was {skip_measured:.0f} s/question; the gate total above")
print("is what that number has to be explained by. A large residual means the cost is")
print("somewhere this cell does not touch (profiler wrapper, retrieval, or run loop).")

## 6b. Calibrate the thresholds — BEFORE the long runs, not after

τ = 0.70/0.70 was fitted to the **3B's** signal distribution. The project's
calibration-non-transfer finding *predicts it will not transfer* to Qwen. If it does
not, and we only find out afterwards, P5 will have spent the night collapsed onto
P1 (retrieve everything) or P3 (retrieve nothing), measuring nothing.

So a **40-question** P5 run harvests Qwen's signal distribution (~20 min), the
thresholds are refitted offline in seconds, and only then do the 200-question runs
start.

**How the refit works — and why not just copy τ, or use the script's p75/p25 hint.**
The sweep script prints "suggested τ_H (entropy p75), τ_M (margin p25)". Those were
the *plan's starting points*, not what the 3B shipped. Measured on the real 3B log:

| | τ_H/τ_M | entropy | margin | probe | ensemble |
|---|---|---|---|---|---|
| shipped 3B | 0.70/0.70 | 52% | 54% | 40% | **50%** |
| p75/p25 hint | 0.919/0.635 | 25% | 25% | 40% | **24%** |

Using p75/p25 would hand Qwen **half** the 3B's retrieval budget, confounding model
scale with how often each model is allowed to retrieve — and the central claim
(*selective beats always-retrieve*) would no longer be comparable across models.

Instead each gate's τ is set so it **fires at the same rate it did on the 3B**. The
retrieval budget is held constant, scale is the only variable, and non-transfer
becomes a sharper claim: not "τ stopped working" but "τ had to move *this far* to
buy the same budget."

The 50 calibration questions are a **stride sample** (every 4th) of the same 200, so
they span every subject. Fitting on a prefix instead overshoots the retrieval budget
by ~6pp — and that error does not shrink with more prefix, because more prefix is
still the same subjects.

*Caveat for the write-up:* the calibration questions are a subset of the evaluation
set, so the operating point is not fitted on held-out data. The 3B was calibrated the
same way (replay over its own run), so the cross-model comparison is consistent —
but state it as a limitation.

In [ ]:
# ~12 min. Harvests gate signals only; this short run's accuracy is not reported.
#
# Calibrates on a STRIDE sample (every 4th of the 200), not the first 50. The
# evaluation set is 200 MMLU questions ordered by subject, so a prefix is one or two
# subjects rather than a sample of the benchmark. Measured on the 3B's own signals,
# a prefix overshoots the retrieval budget by ~6pp and the error does not shrink
# with N; the stride sample lands within ~1pp. See scripts/make_calibration_set.py.
TAG = f'qwen{MODEL}'
CAL_LOG = f'{DRIVE}/logs/p5_{TAG}_calib.jsonl'
CAL_SET = 'data/raw/mirage/calib50.json'
CAL_N = 50

import os, subprocess
if not os.path.exists(CAL_SET):
    subprocess.run(['python', 'scripts/make_calibration_set.py',
                    '--output', CAL_SET, '-n', str(CAL_N)], check=True)

done = sum(1 for _ in open(CAL_LOG)) if os.path.exists(CAL_LOG) else 0
if done < CAL_N:
    cmd = ['python', 'scripts/run_experiment.py',
           '--model-config', spec['cfg'],
           '--policy', 'configs/policies/p5_gated_entropy.yaml',
           '--experiment', 'configs/experiments/mirage_medcorp.yaml',
           '--dataset', CAL_SET,
           '--retrieval-mode', 'hybrid',
           '--bm25-index', BM25, '--faiss-index', FAISS,
           '--max-questions', str(CAL_N),
           '--output', CAL_LOG] + GGUF_OVERRIDE
    subprocess.run(cmd, check=True)
else:
    print(f'calibration log already has {done} records')

!python scripts/run_threshold_sweep.py --logs {CAL_LOG}:QWEN_CAL


In [ ]:
# Calibrate Qwen by matching the 3B's RETRIEVAL BUDGET, not by copying its tau.
#
# Measured on the 3B at its shipped tau=0.70/0.70 (results/raw_logs/p5_medcorp_mcq.jsonl,
# n=200). Hard-coded because results/raw_logs/ is gitignored and so is not present
# in the Colab checkout.
TARGET_ENTROPY_RATE = 0.52   # fraction of queries where the entropy gate votes retrieve
TARGET_MARGIN_RATE  = 0.54   # ditto, margin gate
TARGET_ENSEMBLE     = 0.50   # resulting 3-gate majority rate on the 3B

import sys, shutil, statistics as st, re
sys.path.insert(0, 'scripts')
from run_threshold_sweep import load_signals, _pct, ensemble_rate, PROBE_THRESHOLD

_, sigs = load_signals(f'{CAL_LOG}:QWEN_CAL')
assert sigs, 'no gate signals in the calibration log — the gates did not fire at all'

ent = [s['entropy'] for s in sigs]
mar = [s['margin'] for s in sigs]
prb = [s['hallucination_probe'] for s in sigs]

for nm, xs in (('entropy', ent), ('margin', mar), ('probe', prb)):
    print(f'{nm:8s} min={min(xs):.3f} p25={_pct(xs,0.25):.3f} '
          f'median={st.median(xs):.3f} p75={_pct(xs,0.75):.3f} max={max(xs):.3f}')

# entropy votes retrieve when signal > tau_H, so to fire on the top 52% of queries
# tau_H sits at the 48th percentile. margin votes when signal < tau_M, so it sits
# at the 54th percentile directly.
TAU_H = round(_pct(ent, 1.0 - TARGET_ENTROPY_RATE), 3)
TAU_M = round(_pct(mar, TARGET_MARGIN_RATE), 3)

old = ensemble_rate(sigs, 0.70, 0.70)      # what the 3B's tau does on Qwen
new = ensemble_rate(sigs, TAU_H, TAU_M)    # budget-matched operating point

print(f'\n{"gate":22s} {"3B tau .70/.70":>15s} {"matched " + str(TAU_H) + "/" + str(TAU_M):>20s} {"3B actual":>10s}')
tgt = {'entropy': TARGET_ENTROPY_RATE, 'margin': TARGET_MARGIN_RATE,
       'hallucination_probe': 0.40, 'ensemble': TARGET_ENSEMBLE}
for g in ('entropy', 'margin', 'hallucination_probe', 'ensemble'):
    print(f'{g:22s} {old[g]:>14.0%} {new[g]:>20.0%} {tgt[g]:>10.0%}')

# --- THE FINDING -----------------------------------------------------------
# Non-transfer is now measured as a DISTANCE: how far tau must move to buy the
# same retrieval budget on a different model.
print(f'\ntau shift needed to hold the budget constant: '
      f'entropy 0.70 -> {TAU_H} ({TAU_H - 0.70:+.3f}), '
      f'margin 0.70 -> {TAU_M} ({TAU_M - 0.70:+.3f})')
if old['ensemble'] in (0.0, 1.0):
    print('NON-TRANSFER: CONFIRMED — the 3B operating point is fully degenerate on Qwen.')
else:
    print(f'NON-TRANSFER: the 3B tau still fires here, but at {old["ensemble"]:.0%} '
          f'vs the intended {TARGET_ENSEMBLE:.0%}.')

# --- failure modes to catch BEFORE committing the night --------------------
# The probe has no threshold on MCQ (letter_match: two drafts disagree -> retrieve),
# so it cannot be recalibrated. If a stronger model simply agrees with itself more
# often, the probe goes quiet and majority-2-of-3 reduces to "entropy AND margin" —
# two gates that already agree 82% of the time. That is a real result about gate
# ensembles at scale, but it has to be seen now, not discovered in the results.
if new['hallucination_probe'] < 0.05:
    print(f'\n[WARN] probe votes retrieve on only {new["hallucination_probe"]:.0%} of queries '
          f'(3B: 40%) — effectively silent.\n'
          f'       The ensemble is now entropy+margin alone. Not a bug: 7B self-agrees\n'
          f'       more than 3B. Report it as a finding.')
elif new['hallucination_probe'] > 0.95:
    print(f'\n[WARN] probe votes retrieve on {new["hallucination_probe"]:.0%} of queries — '
          f'always-on, adds no discrimination.')

if not 0.15 < new['ensemble'] < 0.85:
    print(f'\n[WARN] ensemble retrieval {new["ensemble"]:.0%} is near-degenerate; P5 will '
          f'behave much like {"P3" if new["ensemble"] < 0.5 else "P1"}.')

# The probe threshold is held constant ACROSS MODELS — that is what keeps the
# probe comparable as scale changes, and is deliberate.
#
# It is NOT held constant across TASK TYPES. On MCQ the probe is threshold-free
# (letter_match). On open-ended it uses f1_threshold, and leaving that at the
# default 0.7 made it fire on 83-87% of questions for both models, which pushed
# the ensemble to ~57% even after entropy and margin were refit to 50%. Step 6c
# refits it on open-ended signals. See the saturation table there.
print(f'\nprobe f1_threshold {PROBE_THRESHOLD} held across models for MCQ; '
      f'refit for open-ended in step 6c.')

P5_POLICY = 'p5_gated_qwen'
src = 'configs/policies/p5_gated_entropy.yaml'
dst = f'configs/policies/{P5_POLICY}.yaml'
text = open(src, encoding='utf-8').read()
text = re.sub(r'entropy_threshold: [\d.]+', f'entropy_threshold: {TAU_H}', text)
text = re.sub(r'margin_threshold: [\d.]+',  f'margin_threshold: {TAU_M}',  text)
text += (f"\n# Refitted for {spec['cfg']} on {len(sigs)} calibration questions by matching\n"
         f"# the 3B's per-gate retrieval budget (entropy {TARGET_ENTROPY_RATE:.0%}, "
         f"margin {TARGET_MARGIN_RATE:.0%}), NOT by copying its tau.\n"
         f"# The 3B tau 0.70/0.70 would have given {old['ensemble']:.0%} ensemble retrieval\n"
         f"# here; the matched point gives {new['ensemble']:.0%} (3B: {TARGET_ENSEMBLE:.0%}).\n")
open(dst, 'w', encoding='utf-8').write(text)
print(f'\nwrote {dst}  ->  P5_POLICY = {P5_POLICY!r}')

# The repo checkout is wiped when the VM recycles; keep the fitted policy on Drive.
shutil.copy(dst, f'{DRIVE}/logs/{P5_POLICY}.yaml')


## 6c. Calibrate the OPEN-ENDED thresholds — separately from MCQ

Step 6b fits tau on **MCQ** drafts. Those same tau were then used for the open-ended run,
and that is why it degenerated. Measured on the 35 open-ended records from 2026-07-25:

| member | tau | signal min / median / max | fired |
|---|---|---|---|
| entropy (fires when signal > tau) | 0.187 | **0.215** / 0.540 / 1.001 | 100% |
| margin (fires when signal < tau) | 0.899 | 0.658 / 0.769 / **0.854** | 100% |
| probe (fires when draft-F1 < tau) | 0.700 | 0.077 / 0.538 / 0.836 | 82.9% |

Entropy's *minimum* was above its threshold and margin's *maximum* was below its own.
Not one question fell on the other side of either. P5-open collapsed into P1-open, so no
selective-retrieval claim can be made from that run.

Open-ended drafts are long free text; MCQ drafts are a single letter. Their signal
distributions have no reason to coincide, so **gate thresholds are task-specific, not
merely model-specific.**

The probe is refit here too. Replaying the 35 records offline: refitting only entropy and
margin to 50% leaves the ensemble at **57%**, because the probe stays stuck at 83%.
Refitting all three lands it at **46%**. Grid-searching to exactly 50% while holding the
probe does hit target, but only by collapsing the margin gate to 5.7% — gaming the
majority vote rather than calibrating it.

Calibration questions are **stride-sampled**, for the same reason step 6b uses a stride
sample: a prefix is not a sample.

In [ ]:
# Harvest open-ended gate signals on a stride-sampled subset (~40 questions).
#
# The policy file used here is irrelevant to what gets recorded: thresholds change
# the retrieve/skip DECISION, not the signal values, and it is the signals we want.
import json, os, subprocess

OPEN_SRC     = 'data/raw/openqa/pubmedqa_labeled.jsonl'
OPEN_CAL_SET = 'data/raw/openqa/calib40_open.jsonl'
CAL_OPEN_N   = 40

rows = [l for l in open(OPEN_SRC, encoding='utf-8') if l.strip()]
step = len(rows) / CAL_OPEN_N
picked = [rows[int(i * step)] for i in range(CAL_OPEN_N)]
with open(OPEN_CAL_SET, 'w', encoding='utf-8') as fh:
    fh.writelines(picked)
print(f'stride sample: {len(picked)} of {len(rows)} questions, every {step:.1f}th')

CAL_OPEN_LOG = f'{DRIVE}/logs/p5_{TAG}_calib_open.jsonl'
done = sum(1 for _ in open(CAL_OPEN_LOG)) if os.path.exists(CAL_OPEN_LOG) else 0
if done < CAL_OPEN_N:
    print(f'harvesting ({done}/{CAL_OPEN_N} already done)...', flush=True)
    cmd = ['python', 'scripts/run_experiment.py',
           '--model-config', spec['cfg'],
           '--policy', 'configs/policies/p5_gated_entropy.yaml',
           '--experiment', 'configs/experiments/pubmedqa_open.yaml',
           '--dataset', OPEN_CAL_SET,
           '--retrieval-mode', 'hybrid',
           '--bm25-index', BM25, '--faiss-index', FAISS,
           '--max-questions', str(CAL_OPEN_N),
           '--output', CAL_OPEN_LOG] + GGUF_OVERRIDE
    subprocess.run(cmd, check=True)
else:
    print(f'open calibration log already has {done} records')

In [ ]:
# Fit all three members to the target per-gate budget, then write the OPEN policy.
#
# 50% matches the MCQ target, so open-ended and MCQ are comparable within a model.
TARGET_OPEN = 0.50

import json, statistics as st, shutil, yaml

def _open_signals(path):
    out = []
    for line in open(path, encoding='utf-8'):
        if not line.strip():
            continue
        d = json.loads(line)
        m = (((d.get('qvault') or {}).get('gate_details') or {}).get('members') or {})
        e, mg, p = m.get('entropy', {}), m.get('margin', {}), m.get('hallucination_probe', {})
        if e.get('mean_entropy') is None or mg.get('mean_margin') is None:
            continue
        out.append((float(e['mean_entropy']), float(mg['mean_margin']),
                    float(p.get('f1', 0.0))))
    return out

def _q(xs, q):
    xs = sorted(xs)
    return xs[min(len(xs) - 1, max(0, int(round(q * (len(xs) - 1)))))]

def _rates(sig, th, tm, tp, min_votes=2):
    ent = [s[0] > th for s in sig]
    mar = [s[1] < tm for s in sig]
    prb = [s[2] < tp for s in sig]
    ens = [sum(v) >= min_votes for v in zip(ent, mar, prb)]
    n = len(sig)
    return dict(entropy=sum(ent)/n, margin=sum(mar)/n,
                hallucination_probe=sum(prb)/n, ensemble=sum(ens)/n)

sig = _open_signals(CAL_OPEN_LOG)
assert sig, 'no open-ended gate signals in the calibration log'
ent = [s[0] for s in sig]; mar = [s[1] for s in sig]; prb = [s[2] for s in sig]

for nm, xs in (('entropy', ent), ('margin', mar), ('probe f1', prb)):
    print(f'{nm:9s} min={min(xs):.3f} p25={_q(xs,.25):.3f} med={_q(xs,.5):.3f} '
          f'p75={_q(xs,.75):.3f} max={max(xs):.3f}')

# entropy fires above tau -> tau at the (1-target) quantile.
# margin and probe fire BELOW tau -> tau at the target quantile.
TAU_H_O = round(_q(ent, 1.0 - TARGET_OPEN), 3)
TAU_M_O = round(_q(mar, TARGET_OPEN), 3)
TAU_P_O = round(_q(prb, TARGET_OPEN), 3)

old = _rates(sig, TAU_H, TAU_M, 0.70)          # the MCQ-fitted operating point
new = _rates(sig, TAU_H_O, TAU_M_O, TAU_P_O)   # the open-ended one

print(f'\n{"gate":22s}{"MCQ tau":>12s}{"open tau":>12s}{"target":>10s}')
for g in ('entropy', 'margin', 'hallucination_probe', 'ensemble'):
    print(f'{g:22s}{old[g]:>11.0%}{new[g]:>12.0%}{TARGET_OPEN:>10.0%}')

if min(old.values()) == 1.0 or max(old.values()) == 0.0:
    print('\nSATURATION CONFIRMED: the MCQ operating point is degenerate on open-ended.')
if not 0.15 < new['ensemble'] < 0.85:
    print(f'\n[WARN] refitted ensemble {new["ensemble"]:.0%} is still near-degenerate. '
          f'Do NOT start the long run; the thresholds are not the whole story.')
else:
    print(f'\nrefitted ensemble {new["ensemble"]:.0%} — safe to run.')

# Write the open-ended policy. Parsed and re-dumped rather than regex-patched,
# because f1_threshold is nested under gate.hallucination_probe.
P5_POLICY_OPEN = 'p5_gated_qwen_open'
_doc = yaml.safe_load(open('configs/policies/p5_gated_entropy.yaml', encoding='utf-8'))
_doc.setdefault('gate', {})
_doc['gate']['entropy_threshold'] = TAU_H_O
_doc['gate']['margin_threshold'] = TAU_M_O
_doc['gate'].setdefault('hallucination_probe', {})
_doc['gate']['hallucination_probe']['f1_threshold'] = TAU_P_O
_doc['gate']['hallucination_probe']['agreement_mode'] = 'f1_threshold'

_hdr = (f"# {P5_POLICY_OPEN}.yaml - OPEN-ENDED thresholds for {spec['cfg']}.\n"
        f"#\n"
        f"# Fitted on {len(sig)} stride-sampled PubMedQA questions by targeting a\n"
        f"# {TARGET_OPEN:.0%} per-gate retrieval budget, matching the MCQ target.\n"
        f"#\n"
        f"# These differ from the MCQ thresholds because open-ended drafts are long\n"
        f"# free text and MCQ drafts are a single letter: the signal distributions do\n"
        f"# not coincide. Reusing the MCQ tau here made every gate fire on every\n"
        f"# question (ensemble {old['ensemble']:.0%}), collapsing P5 into P1.\n"
        f"#\n"
        f"# The probe f1_threshold IS refit here. Holding it constant across MODELS\n"
        f"# keeps the probe comparable as scale changes; holding it constant across\n"
        f"# TASK TYPES is what caused the saturation.\n")
_dst = f'configs/policies/{P5_POLICY_OPEN}.yaml'
with open(_dst, 'w', encoding='utf-8') as fh:
    fh.write(_hdr + yaml.safe_dump(_doc, sort_keys=False))
print(f'\nwrote {_dst}  ->  P5_POLICY_OPEN = {P5_POLICY_OPEN!r}')
print(f'  entropy {TAU_H} -> {TAU_H_O}   margin {TAU_M} -> {TAU_M_O}   '
      f'probe 0.7 -> {TAU_P_O}')

shutil.copy(_dst, f'{DRIVE}/logs/{P5_POLICY_OPEN}.yaml')

## 7. Run the policies

All four policies, MCQ and open-ended. Logs are written **straight to Drive**, and every run skips question-ids already present — so if Colab disconnects, just re-run this cell and it picks up where it stopped.

Start with P3 and P5: they are the informative pair (P3 sets the new ceiling; P5 tests whether *selective beats always* survives the scale change). P1 and P4 complete the comparison.

In [ ]:
import subprocess, os

MCQ  = dict(dataset='data/raw/mirage/benchmark.json',
            experiment='configs/experiments/mirage_medcorp.yaml')
OPEN = dict(dataset='data/raw/openqa/pubmedqa_labeled.jsonl',
            experiment='configs/experiments/pubmedqa_open.yaml')

assert 'P5_POLICY' in dir(), 'Run the 6b calibration cells first — otherwise P5 ' \
                             'would silently use the 3B thresholds.'
assert 'P5_POLICY_OPEN' in dir(), 'Run the 6c open-ended calibration first — otherwise ' \
                                  'P5-open reuses the MCQ thresholds and every question ' \
                                  'retrieves (see the saturation table in 6c).'

# MCQ and open-ended use SEPARATE P5 policies. Sharing one is what caused the
# 2026-07-25 open-ended run to retrieve on 100% of questions.
#
# Order is deliberate: the two cheap P3 runs finish first, so a disconnect at any
# point still leaves a usable result (the new closed-book ceiling) rather than
# nothing. P5 is the expensive pair and goes last.
#
# The open P5 output is a NEW file. The old p5_{TAG}_open.jsonl holds 35 records
# written at the MCQ thresholds; resuming into it would append records from a
# different operating point to the same log, and that file is also the evidence
# for the saturation finding. Do not overwrite it.
RUNS = [
    ('p3_closed_book', 'none',   'mcq',  MCQ,  f'p3_{TAG}_mcq'),
    ('p3_closed_book', 'none',   'open', OPEN, f'p3_{TAG}_open'),
    (P5_POLICY,        'hybrid', 'mcq',  MCQ,  f'p5_{TAG}_mcq'),
    (P5_POLICY_OPEN,   'hybrid', 'open', OPEN, f'p5_{TAG}_open_v2'),
]

for policy, mode, kind, ds, stem in RUNS:
    out = f'{DRIVE}/logs/{stem}.jsonl'
    done = sum(1 for _ in open(out)) if os.path.exists(out) else 0
    if done >= 200:
        print(f'[skip] {policy} {kind}: already {done}/200')
        continue
    print(f'\n=== {policy} {kind} ({done}/200 done) ===', flush=True)
    cmd = [
        'python', 'scripts/run_experiment.py',
        '--model-config', spec['cfg'],
        '--policy', f'configs/policies/{policy}.yaml',
        '--experiment', ds['experiment'],
        '--dataset', ds['dataset'],
        '--retrieval-mode', mode,
        '--output', out,
    ] + GGUF_OVERRIDE
    if mode != 'none':
        cmd += ['--bm25-index', BM25, '--faiss-index', FAISS]
    subprocess.run(cmd, check=False)   # keep going if one policy dies

print('\nAll runs attempted. Re-run this cell after any disconnect to resume.')


## 8. Confirm the calibration on the full run

The thresholds were fitted in step 6b on 40 questions. This replays the **full 200**
so the write-up can report the complete signal distribution and confirm the subset
was representative. No model calls — seconds.

In [ ]:
# Post-hoc: the full 200-question signal distribution, for the write-up's
# calibration table. The thresholds were already fitted in step 6b; this confirms
# the 40-question subset was representative.
!python scripts/run_threshold_sweep.py \
  --logs {DRIVE}/logs/p5_{TAG}_mcq.jsonl:QWEN_MCQ \
         {DRIVE}/logs/p5_{TAG}_open.jsonl:QWEN_OPEN


## 9. Copy the logs back into the repo

Download `MyDrive/medrag/logs/*.jsonl` and drop them into `results/raw_logs/` on your machine. Then, locally:

```bash
python scripts/generate_tables.py     # numbers.tex picks up the new models
python scripts/generate_figures.py    # fig_scaling.png
python -m pytest tests/               # drift guard confirms text == logs
```

The three questions the curve answers — **each of which is a result whichever way it lands**:

1. Does **selective beats always-retrieve** still hold at 7B and 14B? (generalises the central claim beyond one model)
2. Does **calibration non-transfer** hold prospectively? (τ=0.70 should fail on Qwen)
3. **Where does the curve sit relative to the safety bars, and is it flattening?** If 14B clears the 70% low bar, that is the scale at which low-risk deployment becomes viable. If it does not, the model-capability-ceiling finding is *strengthened*.

## 10. Overnight batch — clean 7B P5-MCQ + full 14B scaling point

Run the setup cells (mount, install, indexes) and the 7B calibration (6b/6c) first, then run this ONE cell and sleep. It is fully resumable — re-run after any disconnect. 14B is smoke-gated: if it OOMs, the 7B clean re-run still completes and nothing you already have is overwritten.

In [ ]:
# ============================================================================
# OVERNIGHT BATCH  —  paste as ONE Colab cell, run, sleep.
#
# Prereqs (the setup cells must have run this session): DRIVE, BM25, FAISS defined;
# the install cell done (llama-cpp GPU wheel + nccl pin + PYTHONPATH); indexes present.
#
# Produces, all resumable (re-run the cell after any disconnect):
#   A) p5_qwen7b_mcq_clean.jsonl        — 7B P5 MCQ with clean latency (energy+mem off)
#   B) p{1,3,5}_qwen14b_{mcq,open}.jsonl — full 14B scaling point + its two calibrations
#
# 14B is SMOKE-GATED: if it OOMs or fails the smoke test, the batch skips 14B and the
# 7B clean re-run still completes. Nothing you already have is touched or overwritten.
# ============================================================================
import os, sys, re, glob, shutil, subprocess
for p in ("src", "scripts"):
    if p not in sys.path:
        sys.path.insert(0, p)
from run_threshold_sweep import load_signals, _pct, ensemble_rate

LOGS = f"{DRIVE}/logs"
os.makedirs(LOGS, exist_ok=True)
MCQ_DS,  MCQ_EXP  = "data/raw/mirage/benchmark.json",        "configs/experiments/mirage_medcorp.yaml"
OPEN_DS, OPEN_EXP = "data/raw/openqa/pubmedqa_labeled.jsonl", "configs/experiments/pubmedqa_open.yaml"
CAL_MCQ = "data/raw/mirage/calib50.json"
BASE_POL = "configs/policies/p5_gated_entropy.yaml"

if not os.path.exists(CAL_MCQ):
    subprocess.run(["python", "scripts/make_calibration_set.py", "--output", CAL_MCQ, "-n", "50"], check=True)

# Restore the 7B policies written by the 6b/6c calibration cells (they live on Drive;
# a fresh VM's repo clone does not have them). Part A needs p5_gated_qwen.yaml.
for src in glob.glob(f"{LOGS}/p5_gated_qwen*.yaml"):
    dst = f"configs/policies/{os.path.basename(src)}"
    if not os.path.exists(dst):
        shutil.copy(src, dst)
        print(f"restored {dst} from Drive", flush=True)
assert os.path.exists("configs/policies/p5_gated_qwen.yaml"), (
    "p5_gated_qwen.yaml not found locally or on Drive — run the 7B calibration (6b) "
    "cell first, or Part A cannot run.")


def _count(path):
    return sum(1 for _ in open(path)) if os.path.exists(path) else 0


def run_exp(cfg, policy, exp, ds, mode, out, n=200):
    done = _count(out)
    if done >= n:
        print(f"    [skip] {os.path.basename(out)} ({done}/{n})", flush=True)
        return
    print(f"    [run ] {os.path.basename(out)} ({done}/{n} done)", flush=True)
    cmd = ["python", "scripts/run_experiment.py", "--model-config", cfg,
           "--policy", policy, "--experiment", exp, "--dataset", ds,
           "--retrieval-mode", mode, "--output", out]
    if mode != "none":
        cmd += ["--bm25-index", BM25, "--faiss-index", FAISS]
    if n != 200:
        cmd += ["--max-questions", str(n)]
    subprocess.run(cmd, check=False)


def setup_model(repo, cfg):
    """Resolve + download the Q4_K_M shards, rewrite gguf_path in the model YAML."""
    from huggingface_hub import list_repo_files, hf_hub_download
    q4 = sorted(f for f in list_repo_files(repo)
                if f.lower().endswith(".gguf") and "q4_k_m" in f.lower())
    assert q4, f"no q4_k_m gguf in {repo}"
    os.makedirs("models", exist_ok=True)
    for shard in q4:
        cached = f"{DRIVE}/models/{os.path.basename(shard)}"
        if not os.path.exists(cached):
            print("    downloading", shard, flush=True)
            hf_hub_download(repo_id=repo, filename=shard, local_dir=f"{DRIVE}/models")
        local = f"models/{os.path.basename(shard)}"
        if not os.path.exists(local):
            os.symlink(os.path.abspath(cached), local)
    gguf = f"models/{os.path.basename(q4[0])}"
    t = open(cfg, encoding="utf-8").read()
    t = re.sub(r"gguf_path: .*", f"gguf_path: {gguf}", t)
    open(cfg, "w", encoding="utf-8").write(t)
    print(f"    {cfg} -> {gguf} ({os.path.getsize(gguf)/1e9:.1f} GB)", flush=True)


def smoke_ok(cfg):
    r = subprocess.run(["python", "scripts/smoke_test_model.py", "--model-config", cfg,
                        "--dataset", MCQ_DS, "--bm25-index", BM25, "--faiss-index", FAISS,
                        "-n", "20"])
    return r.returncode == 0


def calibrate(cfg, tag, task):
    """Budget-match against the 3B's per-gate rates. Reproduces the shipped 7B
    thresholds exactly (validated). Writes a task-specific policy, returns its path."""
    if task == "mcq":
        ds, exp, n, tgt_e, tgt_m = CAL_MCQ, MCQ_EXP, 50, 0.52, 0.54
    else:
        ds, exp, n, tgt_e, tgt_m = OPEN_DS, OPEN_EXP, 40, 0.50, 0.50
    cal_log = f"{LOGS}/p5_{tag}_calib_{task}.jsonl"
    if _count(cal_log) < n:
        run_exp(cfg, BASE_POL, exp, ds, "hybrid", cal_log, n=n)
    _, sigs = load_signals(f"{cal_log}:CAL")
    assert sigs, f"no gate signals in {cal_log}"
    tau_h = round(_pct([s["entropy"] for s in sigs], 1 - tgt_e), 3)
    tau_m = round(_pct([s["margin"]  for s in sigs], tgt_m),     3)
    t = open(BASE_POL, encoding="utf-8").read()
    t = re.sub(r"entropy_threshold: [\d.]+", f"entropy_threshold: {tau_h}", t)
    t = re.sub(r"margin_threshold: [\d.]+",  f"margin_threshold: {tau_m}",  t)
    if task == "open":
        tau_p = round(_pct([s["hallucination_probe"] for s in sigs], 0.50), 3)
        t = t.rstrip() + (f"\n  hallucination_probe:\n    f1_threshold: {tau_p}\n"
                          f"    agreement_mode: f1_threshold\n")
    pol = f"configs/policies/p5_gated_{tag}_{task}.yaml"
    open(pol, "w", encoding="utf-8").write(t)
    shutil.copy(pol, f"{LOGS}/{os.path.basename(pol)}")   # survive VM recycle
    r = ensemble_rate(sigs, tau_h, tau_m)["ensemble"]
    print(f"    calib {tag}/{task}: tau_H={tau_h} tau_M={tau_m} -> ensemble {r:.0%}", flush=True)
    return pol


# ---------- PART A: clean 7B P5 MCQ (energy+memory already off in qwen7b.yaml) ----------
print("=== PART A: clean-latency 7B P5 MCQ ===", flush=True)
run_exp("configs/models/qwen7b.yaml", "configs/policies/p5_gated_qwen.yaml",
        MCQ_EXP, MCQ_DS, "hybrid", f"{LOGS}/p5_qwen7b_mcq_clean.jsonl")

# ---------- PART B: full 14B scaling point ----------
print("\n=== PART B: Qwen-14B ===", flush=True)
CFG14 = "configs/models/qwen14b.yaml"
setup_model("Qwen/Qwen2.5-14B-Instruct-GGUF", CFG14)

if not smoke_ok(CFG14):
    print("    14B smoke FAILED at n_ctx=4096 (likely VRAM). Retrying at n_ctx=2048...", flush=True)
    t = open(CFG14, encoding="utf-8").read()
    t = re.sub(r"n_ctx: \d+", "n_ctx: 2048", t)
    open(CFG14, "w", encoding="utf-8").write(t)

if smoke_ok(CFG14):
    pol_mcq  = calibrate(CFG14, "qwen14b", "mcq")
    pol_open = calibrate(CFG14, "qwen14b", "open")
    P1 = "configs/policies/p1_always_retrieve.yaml"
    P3 = "configs/policies/p3_closed_book.yaml"
    RUNS = [
        (P3,       "none",   "mcq",  MCQ_EXP,  MCQ_DS,  "p3_qwen14b_mcq.jsonl"),
        (P3,       "none",   "open", OPEN_EXP, OPEN_DS, "p3_qwen14b_open.jsonl"),
        (pol_mcq,  "hybrid", "mcq",  MCQ_EXP,  MCQ_DS,  "p5_qwen14b_mcq.jsonl"),
        (pol_open, "hybrid", "open", OPEN_EXP, OPEN_DS, "p5_qwen14b_open.jsonl"),
        (P1,       "hybrid", "mcq",  MCQ_EXP,  MCQ_DS,  "p1_qwen14b_mcq.jsonl"),
        (P1,       "hybrid", "open", OPEN_EXP, OPEN_DS, "p1_qwen14b_open.jsonl"),
    ]
    for pol, mode, kind, exp, ds, out in RUNS:
        print(f"  14B {os.path.basename(pol)} {kind}", flush=True)
        run_exp(CFG14, pol, exp, ds, mode, f"{LOGS}/{out}")
else:
    print("    14B smoke FAILED at n_ctx=2048 too — skipping 14B. Part A result is safe.",
          flush=True)

print("\n=== BATCH DONE. Re-run this cell after any disconnect to resume. ===", flush=True)
